In [1]:
# Cell 1: Imports and deterministic seeds (numpy, torch/TF, PIL, torchvision, sklearn)
# (If you prefer PyTorch, we'll default to PyTorch here)
import os, random, math, json, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fastprogress import progress_bar

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# PyTorch imports
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
import torchvision
from torchvision import transforms
from torchvision import models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
print("Torch", torch.__version__, "CUDA available:", torch.cuda.is_available())


Torch 2.8.0+cu126 CUDA available: False


In [3]:
# Cell 2: Unzip provided archives into workspace and list top files; assert expected counts (assumptions may be adjusted)
# Put your uploaded files into /content or Colab Files (or mount drive and set BASE_DIR)
BASE = Path("/content/mgcls")
BASE.mkdir(parents=True, exist_ok=True)

# If files are in /mnt/data (from your environment), copy them into working dir
UPLOAD_DIR = Path("/mnt")  # change if your files are elsewhere
for fname in ["typ.zip", "exo.zip", "unl.zip", "labels.csv", "test.csv"]:
    src = UPLOAD_DIR / fname
    if src.exists():
        shutil.copy(src, BASE / fname)

# Unzip if present
for z in ["typ.zip", "exo.zip", "unl.zip"]:
    zpath = BASE / z
    if zpath.exists():
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(BASE / z.replace('.zip',''))
    else:
        print("Warning: missing", zpath)

# Quick counts
typ_count = len(list((BASE/"typ").glob("**/*.*"))) if (BASE/"typ").exists() else None
exo_count = len(list((BASE/"exo").glob("**/*.*"))) if (BASE/"exo").exists() else None
unl_count = len(list((BASE/"unl").glob("**/*.*"))) if (BASE/"unl").exists() else None
print("typ_count:", typ_count, "exo_count:", exo_count, "unl_count:", unl_count)
# show top 5 filenames each
for d in ["typ","exo","unl"]:
    p = BASE / d
    if p.exists():
        print(f"Sample from {d}:", [str(x.name) for x in list(p.glob('*'))[:5]])


typ_count: 2049 exo_count: 72 unl_count: 13821
Sample from typ: ['typ_PNG']
Sample from exo: ['exo_PNG']
Sample from unl: ['unl_PNG']


In [4]:
# Cell 3: Load labels.csv and test.csv; show head and unique labels
labels_path = BASE / "labels.csv"
test_path = BASE / "test.csv"
assert labels_path.exists(), f"{labels_path} not found - upload it to /mnt/data or update path"
labels_df = pd.read_csv(labels_path, header=None, names=["ra", "dec", "label", "col4", "col5"])
print("labels.csv shape:", labels_df.shape)
display(labels_df.head(8))

# Clean up unused columns
labels_df = labels_df[["ra", "dec", "label"]].dropna(subset=["ra","dec","label"])
print("After cleanup:", labels_df.shape)
print("Unique labels:", labels_df["label"].unique())

if test_path.exists():
    test_df = pd.read_csv(test_path)
    print("test.csv shape:", test_df.shape)
    display(test_df.head(6))
else:
    print("test.csv not found; proceed later when uploaded")
# Show unique labels (may be multiple rows per source)
if 'label' in labels_df.columns:
    print("Unique labels:", sorted(labels_df['label'].unique())[:30])
else:
    print("Columns in labels.csv:", labels_df.columns.tolist())


labels.csv shape: (2178, 5)


,ra,dec,label,col4,col5
0,10.328221,-20.476357,FR II,NaN,NaN
1,92.109802,-49.431413,typical,NaN,NaN
2,88.916825,-59.431868,Point Source,NaN,NaN
3,5.457981,-25.589637,FR II,NaN,NaN
4,119.417608,-53.396711,FR II,NaN,NaN
5,144.100249,-76.517079,Bent,NaN,NaN
6,212.307137,-42.897881,Bent,NaN,NaN
7,108.108297,-60.170606,Point Source,NaN,NaN


After cleanup: (2178, 3)
Unique labels: ['FR II' 'typical' 'Point Source' 'Bent' 'Should be discarded' 'FR I'
 'Exotic' 'S/Z shaped' 'X-Shaped']
test.csv shape: (99, 2)


,201.7436567,-31.32163727
0,234.261286,-46.590846
1,66.793081,-62.375058
2,108.760518,-59.958776
3,202.148240,-31.432391
4,57.025208,-73.861150
5,216.316836,-54.750811


Unique labels: ['Bent', 'Exotic', 'FR I', 'FR II', 'Point Source', 'S/Z shaped', 'Should be discarded', 'X-Shaped', 'typical']


In [5]:
# ✅ Drop 'Should be discarded' and clean label text
labels_df = labels_df[labels_df["label"].str.lower().str.contains("discard") == False]

# Standardize label casing and spacing
labels_df["label"] = (
    labels_df["label"]
    .str.strip()
    .str.replace("-", " ", regex=False)
    .str.replace("/", " ", regex=False)
    .str.title()
)

# Verify
print("Cleaned labels count:", len(labels_df))
print("Unique cleaned labels:", sorted(labels_df["label"].unique()))
display(labels_df.head(8))


Cleaned labels count: 2048
Unique cleaned labels: ['Bent', 'Exotic', 'Fr I', 'Fr Ii', 'Point Source', 'S Z Shaped', 'Typical', 'X Shaped']


,ra,dec,label
0,10.328221,-20.476357,Fr Ii
1,92.109802,-49.431413,Typical
2,88.916825,-59.431868,Point Source
3,5.457981,-25.589637,Fr Ii
4,119.417608,-53.396711,Fr Ii
5,144.100249,-76.517079,Bent
6,212.307137,-42.897881,Bent
7,108.108297,-60.170606,Point Source


In [6]:
# Inspect structure to see where real images live
!find /content/mgcls -maxdepth 3 -type d


/content/mgcls
/content/mgcls/unl
/content/mgcls/unl/unl_PNG
/content/mgcls/exo
/content/mgcls/exo/exo_PNG
/content/mgcls/exo/exo_PNG/.ipynb_checkpoints
/content/mgcls/typ
/content/mgcls/typ/typ_PNG


In [7]:
# ✅ Final Fixed Cell 4 — recursive image search under typ_PNG/exo_PNG/unl_PNG folders
import re
import pandas as pd
from pathlib import Path

def extract_coords_from_filename(fname):
    """Extract first two floats (RA, Dec) from filename."""
    name = Path(fname).stem
    nums = re.findall(r"-?\d+\.\d+", name)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    # fallback for integer coords (unlikely here)
    nums = re.findall(r"-?\d+", name)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    return None

images = []
for subset in ["typ", "exo"]:
    subset_path = BASE / subset
    # 🔍 recursively search all common image extensions
    for f in subset_path.rglob("*"):
        if f.suffix.lower() in [".png", ".jpg", ".jpeg", ".tif", ".tiff"] and f.is_file():
            coords = extract_coords_from_filename(f.name)
            images.append({
                "filename": str(f),
                "subset": subset,
                "fname": f.name,
                "coords": coords
            })

images_df = pd.DataFrame(images)
print(f"Extracted coords for {len(images_df)} images")
print(images_df.head(8))


Extracted coords for 2120 images
                                            filename subset  \
0  /content/mgcls/typ/typ_PNG/1.104 -24.305_[0.03...    typ   
1  /content/mgcls/typ/typ_PNG/342.416 -44.300_[0....    typ   
2  /content/mgcls/typ/typ_PNG/3.456 -19.552_[0.02...    typ   
3  /content/mgcls/typ/typ_PNG/98.157 -56.098_[0.0...    typ   
4  /content/mgcls/typ/typ_PNG/124.655 -57.361_[0....    typ   
5  /content/mgcls/typ/typ_PNG/86.930 -21.874_[0.0...    typ   
6  /content/mgcls/typ/typ_PNG/99.649 -55.166_[0.0...    typ   
7  /content/mgcls/typ/typ_PNG/229.622 -46.705_[0....    typ   

                                               fname              coords  
0  1.104 -24.305_[0.03117146 0.03117146] deg_(Abe...    (1.104, -24.305)  
1  342.416 -44.300_[0.009 0.009] deg_(Abell_S1063...    (342.416, -44.3)  
2  3.456 -19.552_[0.0241024 0.0241024] deg_(Abell...    (3.456, -19.552)  
3  98.157 -56.098_[0.01570108 0.01570108] deg_(J0...   (98.157, -56.098)  
4  124.655 -57.361_[0.01

In [8]:
# Cell 5: Map label coordinates to closest image file by Euclidean distance in coordinate space
# - Assumes labels.csv has two columns for coords called like 'coord1','coord2' or the first two columns are coords.
lab = labels_df.copy()
# heuristics to find coord columns
cols = lab.columns.tolist()
coord_cols = None
for c0 in cols[:3]:
    for c1 in cols[:3]:
        if c0!=c1:
            try:
                _ = lab[[c0,c1]].astype(float)
                coord_cols = (c0,c1)
                break
            except:
                continue
    if coord_cols:
        break
assert coord_cols is not None, f"Could not find numeric coordinate columns in labels.csv; columns: {cols}"
lab = lab.rename(columns={coord_cols[0]:'ra', coord_cols[1]:'dec'})
lab['ra'] = lab['ra'].astype(float)
lab['dec'] = lab['dec'].astype(float)

# Filter out hidden checkpoint or system files before building KDTree
images_df = images_df[~images_df['filename'].str.contains('.ipynb_checkpoints')]
images_df = images_df[images_df['filename'].str.lower().str.endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
print("Filtered images_df length:", len(images_df))


# Build KDTree on images that had coords extracted
coords_list = [c for c in images_df['coords'] if c is not None]
img_index = images_df[images_df['coords'].notnull()].reset_index(drop=True)
from scipy.spatial import cKDTree
tree = cKDTree(np.vstack(img_index['coords'].values))
matches=[]
for i,row in lab.iterrows():
    r,d = row['ra'], row['dec']
    dist, idx = tree.query([r,d], k=1)
    matched = img_index.loc[idx, 'filename']
    matches.append(matched)
lab['matched_filename'] = matches
display(lab.head(8))
# merge to get label lists per filename
grouped = lab.groupby('matched_filename')['label'].apply(lambda x: list(x.unique())).reset_index()
grouped.head(6)


Filtered images_df length: 2107


,ra,dec,label,matched_filename
0,10.328221,-20.476357,Fr Ii,/content/mgcls/typ/typ_PNG/10.328 -20.476_[0.0...
1,92.109802,-49.431413,Typical,/content/mgcls/typ/typ_PNG/92.110 -49.431_[0.0...
2,88.916825,-59.431868,Point Source,/content/mgcls/typ/typ_PNG/88.917 -59.432_[0.0...
3,5.457981,-25.589637,Fr Ii,/content/mgcls/typ/typ_PNG/5.458 -25.590_[0.00...
4,119.417608,-53.396711,Fr Ii,/content/mgcls/typ/typ_PNG/119.418 -53.397_[0....
5,144.100249,-76.517079,Bent,/content/mgcls/typ/typ_PNG/144.100 -76.517_[0....
6,212.307137,-42.897881,Bent,/content/mgcls/typ/typ_PNG/212.307 -42.898_[0....
7,108.108297,-60.170606,Point Source,/content/mgcls/typ/typ_PNG/108.108 -60.171_[0....


,matched_filename,label
0,/content/mgcls/exo/exo_PNG/0.523 -24.568_[0.04...,[Exotic]
1,/content/mgcls/exo/exo_PNG/1.028 -25.064_[0.07...,[Exotic]
2,/content/mgcls/exo/exo_PNG/1.076 -24.429_[0.03...,[Exotic]
3,/content/mgcls/exo/exo_PNG/1.448 -24.492_[0.04...,[Exotic]
4,/content/mgcls/exo/exo_PNG/108.614 -60.373_[0....,[Bent]
5,/content/mgcls/exo/exo_PNG/116.858 -53.535_[0....,[S Z Shaped]


In [9]:
# Cell 6: Create final labeled dataframe with multi-label binarizer and show class counts
merged = img_index.merge(grouped, left_on='filename', right_on='matched_filename', how='left')
merged['label'] = merged['label'].apply(lambda x: x if isinstance(x,list) else [])
mlb = MultiLabelBinarizer(sparse_output=False)
Y = mlb.fit_transform(merged['label'])
label_classes = mlb.classes_.tolist()
print("Classes:", label_classes)
class_counts = Y.sum(axis=0)
print("Class counts:\n", dict(zip(label_classes, class_counts)))
merged['labels_list'] = merged['label']
merged['multi_hot'] = list(Y)
display(merged.head(6))
# assertion: at least 50 labeled examples exist
assert merged.shape[0] > 10, "Too few labeled images found; check mapping step."


Classes: ['Bent', 'Exotic', 'Fr I', 'Fr Ii', 'Point Source', 'S Z Shaped', 'Typical', 'X Shaped']
Class counts:
 {'Bent': np.int64(423), 'Exotic': np.int64(18), 'Fr I': np.int64(428), 'Fr Ii': np.int64(674), 'Point Source': np.int64(429), 'S Z Shaped': np.int64(17), 'Typical': np.int64(24), 'X Shaped': np.int64(5)}


,filename,subset,fname,coords,matched_filename,label,labels_list,multi_hot
0,/content/mgcls/typ/typ_PNG/1.104 -24.305_[0.03...,typ,1.104 -24.305_[0.03117146 0.03117146] deg_(Abe...,"(1.104, -24.305)",/content/mgcls/typ/typ_PNG/1.104 -24.305_[0.03...,[Fr Ii],[Fr Ii],"[0, 0, 0, 1, 0, 0, 0, 0]"
1,/content/mgcls/typ/typ_PNG/342.416 -44.300_[0....,typ,342.416 -44.300_[0.009 0.009] deg_(Abell_S1063...,"(342.416, -44.3)",/content/mgcls/typ/typ_PNG/342.416 -44.300_[0....,[Point Source],[Point Source],"[0, 0, 0, 0, 1, 0, 0, 0]"
2,/content/mgcls/typ/typ_PNG/3.456 -19.552_[0.02...,typ,3.456 -19.552_[0.0241024 0.0241024] deg_(Abell...,"(3.456, -19.552)",/content/mgcls/typ/typ_PNG/3.456 -19.552_[0.02...,[Point Source],[Point Source],"[0, 0, 0, 0, 1, 0, 0, 0]"
3,/content/mgcls/typ/typ_PNG/98.157 -56.098_[0.0...,typ,98.157 -56.098_[0.01570108 0.01570108] deg_(J0...,"(98.157, -56.098)",/content/mgcls/typ/typ_PNG/98.157 -56.098_[0.0...,[Bent],[Bent],"[1, 0, 0, 0, 0, 0, 0, 0]"
4,/content/mgcls/typ/typ_PNG/124.655 -57.361_[0....,typ,124.655 -57.361_[0.01449168 0.01449168] deg_(J...,"(124.655, -57.361)",/content/mgcls/typ/typ_PNG/124.655 -57.361_[0....,[Fr I],[Fr I],"[0, 0, 1, 0, 0, 0, 0, 0]"
5,/content/mgcls/typ/typ_PNG/86.930 -21.874_[0.0...,typ,86.930 -21.874_[0.01169882 0.01169882] deg_(Ab...,"(86.93, -21.874)",NaN,[],[],"[0, 0, 0, 0, 0, 0, 0, 0]"


In [11]:
# Cell 7: Train/val split (deterministic) and small DataLoader definition for debugging
from torch.utils.data import Dataset, DataLoader
IMG_SIZE = 224
BATCH = 32

# Basic transforms for debugging — real training uses augmentations
basic_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

class RadioDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filename']).convert('RGB')
        if self.transform: img = self.transform(img)
        target = torch.tensor(row['multi_hot'], dtype=torch.float32)
        return img, target

train_df, val_df = train_test_split(merged, test_size=0.15, random_state=SEED, stratify=None)
print("train:", len(train_df), "val:", len(val_df))
# quick loader
train_loader = DataLoader(RadioDataset(train_df, basic_transform), batch_size=min(BATCH,len(train_df)), shuffle=True)
imgs, targets = next(iter(train_loader))
print("Batch imgs:", imgs.shape, "targets:", targets.shape)


train: 1790 val: 317
Batch imgs: torch.Size([32, 3, 224, 224]) targets: torch.Size([32, 8])


In [12]:
# Cell 8: Build a simple transfer-learning model (EfficientNet or ResNet) for multi-label classification
import torch.nn as nn

n_classes = len(label_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# using resnet50 as default
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# replace final layer
in_features = backbone.fc.in_features
backbone.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, n_classes)
)
model = backbone.to(device)
print("Model ready with output dim", n_classes)
# quick forward pass to ensure no error
model.eval()
with torch.no_grad():
    out = model(imgs.to(device))
    print("Forward output shape:", out.shape)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 164MB/s]


Model ready with output dim 8
Forward output shape: torch.Size([32, 8])


In [ ]:
# Cell 9: Training loop (single epoch quick run) with assertions and checkpoint save
# - This is a short smoke-test training loop; for full training set epochs increase epochs.
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = GradScaler()
EPOCHS = 1

def train_one_epoch(loader, model, opt, crit, device, scaler):
    model.train()
    total_loss = 0.0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        with autocast():
            logits = model(x)
            loss = crit(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

history = {}
history['train_loss'] = train_one_epoch(train_loader, model, optimizer, criterion, device, scaler)
print("Train loss (smoke):", history['train_loss'])
# save smoke checkpoint
torch.save(model.state_dict(), "/content/model_smoke.pth")
print("Saved /content/model_smoke.pth")


In [ ]:
# Cell 10: Inference on unlabeled images — produce probabilities and prepare for pseudo-labelling
# - Runs model in eval mode on unlabeled dir; writes CSV with top probabilities
model.eval()
unl_dir = BASE / "unl"
unl_files = sorted([str(f) for f in unl_dir.glob("*")])
print("Unlabeled files count:", len(unl_files))
unl_transform = basic_transform
batch_size = 64
preds = []
filenames = []
with torch.no_grad():
    for i in range(0, len(unl_files), batch_size):
        batch_files = unl_files[i:i+batch_size]
        imgs = []
        for f in batch_files:
            imgs.append(unl_transform(Image.open(f).convert('RGB')))
        imgs = torch.stack(imgs).to(device)
        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds.append(probs)
        filenames.extend(batch_files)
preds = np.vstack(preds)
df_unl_preds = pd.DataFrame(preds, columns=label_classes)
df_unl_preds['filename'] = filenames
df_unl_preds.to_csv("/content/unl_preds.csv", index=False)
print("Saved /content/unl_preds.csv with shape", df_unl_preds.shape)
display(df_unl_preds.head())


In [ ]:
# Cell 11: Pseudo-label selection — apply a confidence threshold and create generated_labels.csv
# - Thresholds per class can be tuned; default global 0.95 for high precision
THRESH = 0.95
selected = []
for idx,row in df_unl_preds.iterrows():
    probs = row[label_classes].values
    high_idx = np.where(probs >= THRESH)[0]
    if len(high_idx)>0:
        labels = [label_classes[i] for i in high_idx]
        # extract coords from filename to create output format
        fname = Path(row['filename']).name
        coords = extract_coords_from_filename(fname)
        if coords is None:
            coord1 = coord2 = ""
        else:
            coord1, coord2 = coords
        selected.append({'coord1':coord1, 'coord2':coord2, 'labels':";".join(labels), 'filename': fname})
pseudo_df = pd.DataFrame(selected)
print("Pseudo-labelled rows:", len(pseudo_df))
pseudo_df.head()
# Save final (but this is just high-confidence subset). Full generated_labels.csv must include every unlabeled file eventually.
pseudo_df.to_csv("/content/pseudo_labels_highconf.csv", index=False)
print("Saved /content/pseudo_labels_highconf.csv")
